# Module 13: Statistics for Data — Solutions

Complete solutions to all exercises.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_california_housing
from sklearn.feature_selection import SelectKBest, f_regression
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Setup complete')

### Solution 1: Descriptive Statistics

In [ ]:
iris = sns.load_dataset('iris')
numeric_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

print('=== Five-Number Summary ===')
five_num = iris[numeric_cols].describe(percentiles=[0.25, 0.5, 0.75]).T
print(five_num[['min', '25%', '50%', '75%', 'max']])

variances = iris[numeric_cols].var()
print('\n=== Variances ===')
print(variances)
print(f'\nHighest variance feature: {variances.idxmax()} with variance {variances.max():.4f}')
print('This implies the widest spread of values, indicating strong discriminative power.')

### Solution 2: Central Limit Theorem

In [ ]:
np.random.seed(42)
population = np.random.uniform(0, 10, 100000)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sample_sizes = [2, 10, 50]

for i, n in enumerate(sample_sizes):
    sample_means = [np.mean(np.random.choice(population, n)) for _ in range(2000)]
    axes[i].hist(sample_means, bins=40, density=True, alpha=0.7, color='steelblue')
    axes[i].set_title(f'Sample Means (n={n})')
    axes[i].axvline(np.mean(population), color='red', linestyle='--', label=f'Pop Mean: {np.mean(population):.2f}')
    axes[i].legend()

plt.tight_layout()
plt.show()
print('As n increases, the sampling distribution becomes more normal (CLT in action).')

### Solution 3: Hypothesis Testing (Titanic Fare)

In [ ]:
titanic = sns.load_dataset('titanic')
titanic = titanic.dropna(subset=['fare', 'survived'])

survived_fare = titanic[titanic['survived'] == 1]['fare']
not_survived_fare = titanic[titanic['survived'] == 0]['fare']

t_stat, p_val = stats.ttest_ind(survived_fare, not_survived_fare, equal_var=False)

print('=== Two-Sample t-test: Fare by Survival ===')
print(f'Survivors - Mean fare: {survived_fare.mean():.2f}, Std: {survived_fare.std():.2f}')
print(f'Non-survivors - Mean fare: {not_survived_fare.mean():.2f}, Std: {not_survived_fare.std():.2f}')
print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {p_val:.6f}')
print('Conclusion: p < 0.05, significant difference in fare between survivors and non-survivors.')

### Solution 4: ANOVA on California Housing

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

df['income_group'] = pd.qcut(df['MedInc'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

groups = [df[df['income_group'] == g]['MedHouseVal'] for g in ['Q1', 'Q2', 'Q3', 'Q4']]
f_stat, p_val = stats.f_oneway(*groups)

print('=== ANOVA: MedHouseVal by Income Quartile ===')
print(f'F-statistic: {f_stat:.4f}')
print(f'p-value: {p_val:.10f}')
print('Conclusion: Highly significant — income strongly predicts house value.')

### Solution 5: Chi-Square Test

In [ ]:
titanic = sns.load_dataset('titanic').dropna(subset=['pclass', 'survived'])
contingency = pd.crosstab(titanic['pclass'], titanic['survived'])
print('Contingency Table:')
print(contingency)

chi2, p, dof, expected = stats.chi2_contingency(contingency)

print(f'\nChi-square: {chi2:.4f}')
print(f'p-value: {p:.10f}')
print(f'Degrees of freedom: {dof}')
print('Conclusion: Passenger class and survival are NOT independent.')

### Solution 6: Feature Selection

In [ ]:
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

selector = SelectKBest(score_func=f_regression, k=3)
X_selected = selector.fit_transform(X, y)

scores = pd.DataFrame({
    'feature': X.columns,
    'f_score': selector.scores_,
    'p_value': selector.pvalues_
}).sort_values('f_score', ascending=False)

print('=== Top Features by F-Score ===')
print(scores)
print(f'\nSelected top 3: {X.columns[selector.get_support()].tolist()}')

### Solution 7: Correlation Analysis

In [ ]:
pearson = df.corr(method='pearson')
spearman = df.corr(method='spearman')

diff = (pearson - spearman).abs()
print('=== Largest Pearson-Spearman Differences ===')
# Upper triangle only
mask = np.triu(np.ones_like(diff, dtype=bool), k=1)
diff_triu = diff.where(mask)
top_diffs = diff_triu.unstack().dropna().sort_values(ascending=False).head(5)
print(top_diffs)
print('\nLarge differences suggest non-linear/monotonic relationships.')

### Solution 8: Bayes Theorem

In [ ]:
def bayes_spam(prior_spam, sensitivity, false_positive_rate):
    p_positive = prior_spam * sensitivity + (1 - prior_spam) * false_positive_rate
    posterior = (prior_spam * sensitivity) / p_positive
    return posterior

result = bayes_spam(0.10, 0.95, 0.02)
print('=== Spam Filter Bayes Calculation ===')
print(f'P(Spam | Flagged): {result:.2%}')
print('Interpretation: Even with a positive flag, only {:.1f}% chance it is spam due to low base rate.'.format(result*100))

### Solution 9: Effect Size

In [ ]:
# From solution 3
mean1 = survived_fare.mean()
mean2 = not_survived_fare.mean()
std1 = survived_fare.std()
std2 = not_survived_fare.std()

pooled_std = np.sqrt((std1**2 + std2**2) / 2)
cohens_d = (mean1 - mean2) / pooled_std

print('=== Cohen\'s d for Fare by Survival ===')
print(f'Cohen\'s d: {cohens_d:.3f}')
if abs(cohens_d) < 0.2:
    print('Effect size: small')
elif abs(cohens_d) < 0.5:
    print('Effect size: medium')
else:
    print('Effect size: large')

### Solution 10: Pairwise T-Tests on Iris

In [ ]:
iris = sns.load_dataset('iris')
species_pairs = [('setosa', 'versicolor'), ('setosa', 'virginica'), ('versicolor', 'virginica')]
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

pval_matrix = pd.DataFrame(index=features, columns=['setosa-vs-versicolor', 'setosa-vs-virginica', 'versicolor-vs-virginica'])

for (s1, s2) in species_pairs:
    col_name = f'{s1}-vs-{s2}'
    for feat in features:
        _, p_val = stats.ttest_ind(iris[iris['species']==s1][feat], iris[iris['species']==s2][feat])
        pval_matrix.loc[feat, col_name] = p_val

pval_matrix = pval_matrix.astype(float)
print('=== P-Values for All Pairwise T-Tests ===')
print(pval_matrix)

plt.figure(figsize=(8, 4))
sns.heatmap(-np.log10(pval_matrix), annot=True, fmt='.1f', cmap='viridis',
            cbar_kws={'label': '-log10(p-value)'})
plt.title('Pairwise T-Test: -log10(p-values)\n(Higher = More Significant)')
plt.tight_layout()
plt.show()
print('\nPetal length and petal width best separate all species pairs.')